# Bias Node Research Notebook

Generic notebook for testing bias nodes and generating comprehensive reports.

## How to Use:
1. **Edit the `BIAS_NODES_TO_TEST` list** in the configuration cell below
2. Add/remove any bias nodes you want to test
3. Run all cells to generate:
   - Feature analysis reports (decile plots, distributions, permutation tests)
   - QuantStats HTML tearsheets (Sharpe, Sortino, drawdowns, etc.)

## Output:
- Reports saved to: `research/reports/bias_nodes/{node_name}/`
- Tearsheets saved to: `research/reports/bias_nodes/{node_name}/{node_name}_tearsheet.html`

In [1]:
import sys
from pathlib import Path

# Get project root
project_root = Path.cwd() if (Path.cwd() / 'vault').exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

# Create reports directory
reports_dir = project_root / 'research' / 'reports' / 'bias_nodes'
reports_dir.mkdir(parents=True, exist_ok=True)
print(f"Reports will be saved to: {reports_dir}")

Reports will be saved to: c:\Users\adabla\Trading-Algo\research\reports\bias_nodes


In [2]:
from utils.enums import Ticker, TimeFrame, Direction
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from feature_extraction.feature_extractor import extract_features_for_bias_node
from feature_selection.base_models import BaseModel, QuantileBinningModel, TwoBinBinningModel
from eda.feature_explorer import FeatureExplorer
from metrics.plotting.graphing.quantstats_reports import generate_tearsheet
import utils.helpers as helpers

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

c:\Users\adabla\Trading-Algo\utils\permutation_test\permute_bars.py:38: UserWarning: feature_selection.opt_thresh not found. Using simplified threshold optimization. For full functionality, ensure the opt_thresh module is available.
  warnings.warn(


---
## Configuration

**Edit the `BIAS_NODES_TO_TEST` list below to add/remove bias nodes.**

Each entry is a dict with:
- `name`: Human-readable name for reports
- `spec`: Bias node specification (module_name, timeframes, params)
- `is_continuous`: True for continuous features (0-100), False for rule-based (0/1)

In [3]:
# =============================================================================
# CONFIGURATION - Edit this cell to customize your research
# =============================================================================

# Tickers to test on (ensemble)
ENSEMBLE_TICKERS = [Ticker.ES, Ticker.NQ, Ticker.YM, Ticker.RTY]

# Date range for analysis
START_DATE = datetime(2000, 1, 1)
END_DATE = datetime(2024, 12, 31)

# Permutation test repetitions (higher = more accurate p-values but slower)
PERMUTATION_REPS = 100

# Generate QuantStats tearsheets (set to False to skip)
GENERATE_TEARSHEETS = True

# =============================================================================
# CACHING CONFIGURATION (NEW)
# =============================================================================
# Set USE_CACHE = True to use pre-computed cached values (~50-100x faster)
# First run with POPULATE_CACHE = True to create caches, then set to False
USE_CACHE = True
POPULATE_CACHE = False  # Set True on first run to populate caches

# =============================================================================
# BIAS NODES TO TEST - Add/remove entries as needed
# =============================================================================
# Each entry needs:
#   - 'name': Display name for reports
#   - 'spec': {'module_name': ..., 'timeframes': [...], 'params': {...}}
#   - 'is_continuous': True for continuous (use QuantileBinning), False for rule-based (TwoBinBinning)

BIAS_NODES_TO_TEST = [
    # ---------------------------------------------------------------------
    # CONTINUOUS FEATURES (0-100 or similar range) - Use QuantileBinning
    # ---------------------------------------------------------------------
    {
        'name': 'Adaptive_RSI',
        'spec': {
            'module_name': 'adaptive_rsi',
            'timeframes': [TimeFrame.D],
            'params': {'min_period': [2, 3], 'max_period': [10, 14], 'volatility_period': [20]}
        },
        'is_continuous': True
    },
    {
        'name': 'Casey_C',
        'spec': {
            'module_name': 'casey_c',
            'timeframes': [TimeFrame.D],
            'params': {'change_period': [5, 10], 'ranking_period': [50, 100]}
        },
        'is_continuous': True
    },
    {
        'name': 'Cyclical_RSI',
        'spec': {
            'module_name': 'cyclical_rsi',
            'timeframes': [TimeFrame.D],
            'params': {'short_period': [5], 'long_period': [20], 'rsi_period': [14]}
        },
        'is_continuous': True
    },
    {
        'name': 'Detrended_RSI',
        'spec': {
            'module_name': 'detrended_rsi',
            'timeframes': [TimeFrame.D],
            'params': {'rsi_period': [14], 'detrend_period': [50, 100]}
        },
        'is_continuous': True
    },
    {
        'name': 'DeMark_REI',
        'spec': {
            'module_name': 'demark_rei',
            'timeframes': [TimeFrame.D],
            'params': {'period': [5, 8, 13]}
        },
        'is_continuous': True
    },
    {
        'name': 'RSI_Percentile',
        'spec': {
            'module_name': 'rsi_percentile',
            'timeframes': [TimeFrame.D],
            'params': {'rsi_period': [14], 'percentile_period': [126, 252]}
        },
        'is_continuous': True
    },
    {
        'name': 'Stochastic_RSI',
        'spec': {
            'module_name': 'stochastic_rsi',
            'timeframes': [TimeFrame.D],
            'params': {'rsi_period': [14], 'stoch_period': [14], 'smooth_k': [3]}
        },
        'is_continuous': True
    },
    {
        'name': 'TSI',
        'spec': {
            'module_name': 'tsi',
            'timeframes': [TimeFrame.D],
            'params': {'long_period': [25], 'short_period': [13]}
        },
        'is_continuous': True
    },
    {
        'name': 'ZScore_RSI',
        'spec': {
            'module_name': 'zscore_rsi',
            'timeframes': [TimeFrame.D],
            'params': {'rsi_period': [14], 'zscore_period': [50, 100]}
        },
        'is_continuous': True
    },
    
    # ---------------------------------------------------------------------
    # RULE-BASED FEATURES (0/1 or -1/0/1 signals) - Use TwoBinBinning
    # ---------------------------------------------------------------------
    {
        'name': 'Percent_B',
        'spec': {
            'module_name': 'percent_b',
            'timeframes': [TimeFrame.D],
            'params': {'period': [20], 'std_dev': [2.0], 'lower_threshold': [0.0, 0.2]}
        },
        'is_continuous': False
    },
    {
        'name': 'Double_7s',
        'spec': {
            'module_name': 'double7s',
            'timeframes': [TimeFrame.D],
            'params': {'short_period': [7], 'ma_period': [200]}
        },
        'is_continuous': False
    },
    {
        'name': 'RSI_Signal',
        'spec': {
            'module_name': 'rsi_signal',
            'timeframes': [TimeFrame.D],
            'params': {'rsi_period': [14], 'oversold': [30], 'overbought': [70], 'mode': ['threshold']}
        },
        'is_continuous': False
    },
    {
        'name': 'SuperTrend_Cross',
        'spec': {
            'module_name': 'supertrend_cross',
            'timeframes': [TimeFrame.D],
            'params': {'atr_period': [10], 'fast_multiplier': [2.0], 'slow_multiplier': [3.0]}
        },
        'is_continuous': False
    },
]

print(f"Configuration loaded:")
print(f"  Tickers: {[t.name for t in ENSEMBLE_TICKERS]}")
print(f"  Date range: {START_DATE.date()} to {END_DATE.date()}")
print(f"  Nodes to test: {len(BIAS_NODES_TO_TEST)}")
print(f"  Generate tearsheets: {GENERATE_TEARSHEETS}")
print(f"  USE_CACHE: {USE_CACHE}")
print(f"  POPULATE_CACHE: {POPULATE_CACHE}")

Configuration loaded:
  Tickers: ['ES', 'NQ', 'YM', 'RTY']
  Date range: 2000-01-01 to 2024-12-31
  Nodes to test: 13
  Generate tearsheets: True
  USE_CACHE: True
  POPULATE_CACHE: False


In [4]:
# =============================================================================
# CACHE POPULATION (Run once to create caches, then set POPULATE_CACHE=False)
# =============================================================================
import time
from itertools import product

if POPULATE_CACHE:
    from utils.cache_manager import CacheManager
    
    print("="*70)
    print("POPULATING CACHES (this takes time but only needs to be done once)")
    print("="*70)
    
    # Point to correct candle data directory
    manager = CacheManager(candle_dir=str(project_root / 'data' / 'ohlc_data'))
    
    # Build list of all bias node specs to cache (expand parameter grids)
    all_specs = []
    for node_config in BIAS_NODES_TO_TEST:
        spec = node_config['spec']
        module_name = spec['module_name']
        timeframes = spec.get('timeframes', [TimeFrame.D])
        params = spec.get('params', {})
        
        # Expand parameter grid
        keys = list(params.keys())
        values = [v if isinstance(v, list) else [v] for v in params.values()]
        if keys:
            param_combos = [dict(zip(keys, combo)) for combo in product(*values)]
        else:
            param_combos = [{}]
        
        for param_combo in param_combos:
            all_specs.append({
                'module_name': module_name,
                'params': param_combo,
                'timeframes': timeframes
            })
    
    # Also need ATR and EWSD for volatility scaling
    all_specs.append({'module_name': 'atr', 'params': {'period': 252}, 'timeframes': [TimeFrame.D]})
    all_specs.append({'module_name': 'ewsd', 'params': {}, 'timeframes': [TimeFrame.D]})
    
    print(f"  Total bias node specs to cache: {len(all_specs)}")
    print(f"  Tickers: {[t.name for t in ENSEMBLE_TICKERS]}")
    print(f"  Total cache files: {len(all_specs) * len(ENSEMBLE_TICKERS)}")
    
    t0 = time.time()
    result = manager.populate_cache(
        bias_node_specs=all_specs,
        tickers=ENSEMBLE_TICKERS,
        start_date=START_DATE,
        end_date=END_DATE,
        show_progress=True,
        overwrite_existing=False  # Skip existing caches
    )
    elapsed = time.time() - t0
    
    print(f"\n{'='*70}")
    print(f"CACHE POPULATION COMPLETE")
    print(f"  Success: {result['success']}/{result['total']}")
    print(f"  Skipped (already cached): {result['skipped']}")
    print(f"  Failed: {result['failed']}")
    print(f"  Time: {elapsed:.1f}s")
    print(f"{'='*70}")
else:
    print("POPULATE_CACHE=False - Skipping cache population")
    print("Set POPULATE_CACHE=True and rerun to create caches")

POPULATE_CACHE=False - Skipping cache population
Set POPULATE_CACHE=True and rerun to create caches


---
## Helper Functions

In [5]:
def test_bias_node(bias_spec: dict, node_name: str, is_continuous: bool = True,
                   show_plots: bool = False, export: bool = True, use_cache: bool = False):
    """
    Test a bias node and generate summary report.
    
    Parameters
    ----------
    use_cache : bool, default=False
        If True, load features from BiasNodeCache instead of streaming candles.
        This provides ~50-100x speedup. Cache must be pre-populated.
    
    Returns:
        results: Dictionary with analysis results, or None if failed
    """
    print(f"\n{'='*70}")
    print(f"Testing: {node_name} {'[CACHED]' if use_cache else '[STREAMING]'}")
    print(f"{'='*70}")
    
    # Extract features
    print("\n[1/3] Extracting features...")
    try:
        t0 = time.time()
        features_df, targets_df = extract_features_for_bias_node(
            bias_spec=bias_spec,
            ticker=ENSEMBLE_TICKERS,
            start=START_DATE,
            end=END_DATE,
            use_millisecond_offset=True,
            target_col='log_return',
            use_cache=use_cache
        )
        extract_time = time.time() - t0
        print(f"  [OK] Features shape: {features_df.shape}")
        print(f"  [OK] Targets shape: {targets_df.shape}")
        feature_cols = [col for col in features_df.columns if col != 'ticker']
        print(f"  [OK] Feature columns: {len(feature_cols)}")
        print(f"  [OK] Extraction time: {extract_time:.2f}s")
    except Exception as e:
        print(f"  [FAIL] Feature extraction failed: {e}")
        import traceback
        traceback.print_exc()
        return None
    
    # Build metadata
    print("\n[2/3] Building metadata...")
    metadata = {'feature_metadata': {}}
    for feature_col in features_df.columns:
        if feature_col != 'ticker':
            parsed = helpers.parse_feature_column_name(feature_col)
            metadata['feature_metadata'][feature_col] = {
                'module': parsed.get('module'),
                'parameters': parsed.get('params', {}),
                'timeframes': [parsed.get('tf')] if parsed.get('tf') else [],
                'base_name': parsed.get('module'),
                'full_name': feature_col
            }
    
    # Create FeatureExplorer
    explorer = FeatureExplorer(
        features_df=features_df,
        targets_df=targets_df,
        metadata=metadata
    )
    print(f"  [OK] FeatureExplorer created with {explorer.n_features} features, {explorer.n_samples} samples")
    
    # Select binning model
    if is_continuous:
        binning_model = QuantileBinningModel(n_bins=10, selection_metric='sortino', strategy='long')
    else:
        binning_model = TwoBinBinningModel(selection_metric='sortino', strategy='long')
    
    # Generate summary report
    print("\n[3/3] Generating summary report...")
    export_path = str(reports_dir / node_name.lower().replace(' ', '_').replace('%', 'pct'))
    
    try:
        results = explorer.generate_summary_report(
            binning_model=binning_model,
            target_col='log_return',
            strategy='long',
            show_plots=show_plots,
            verbose=False,
            export_report=export,
            export_path=export_path if export else None,
            permutation_test_nreps=PERMUTATION_REPS,
            include_ticker_plots=False
        )
        print(f"  [OK] Report generated successfully")
        if export:
            print(f"  [OK] Report exported to: {export_path}")
        
        # Print key metrics (key is 'permutation_test', not 'permutation_results')
        if results and 'permutation_test' in results and results['permutation_test'] is not None:
            perm_df = results['permutation_test']
            if len(perm_df) > 0:
                print(f"\n  Permutation Test Results:")
                for _, row in perm_df.iterrows():
                    pval = row.get('pval', row.get('p_value', 'N/A'))
                    sig = '[SIG]' if pval <= 0.1 else '[---]'
                    feature = row.get('feature', 'unknown')[:50]
                    print(f"    {sig} {feature}: p={pval:.4f}")
        
        # Store features_df and targets_df in results for tearsheet generation
        results['features_df'] = features_df
        results['targets_df'] = targets_df
        results['binning_model'] = binning_model
        results['extract_time'] = extract_time
        
        return results
        
    except Exception as e:
        print(f"  [FAIL] Report generation failed: {e}")
        import traceback
        traceback.print_exc()
        return None


def _get_best_feature_from_permutation(perm_df: pd.DataFrame) -> str:
    """
    Get the best feature name from permutation test results.
    Returns the feature with the lowest p-value.
    """
    if perm_df is None or len(perm_df) == 0:
        return None
    
    pval_col = 'pval' if 'pval' in perm_df.columns else 'p_value'
    best_row = perm_df.loc[perm_df[pval_col].idxmin()]
    return best_row['feature']


def generate_node_tearsheet(bias_spec: dict, node_name: str, is_continuous: bool = True,
                            perm_df: pd.DataFrame = None, features_df: pd.DataFrame = None,
                            targets_df: pd.DataFrame = None):
    """
    Generate a QuantStats HTML tearsheet for a bias node.

    Uses the pre-extracted features from feature analysis to ensure consistency.
    Trains on full ensemble, displays ES-only tearsheet.

    Parameters
    ----------
    bias_spec : dict
        Bias node specification (not used directly, kept for API compatibility)
    node_name : str
        Name of the node for display
    is_continuous : bool
        Whether feature is continuous (for binning model selection)
    perm_df : pd.DataFrame
        Permutation test results (to select best feature)
    features_df : pd.DataFrame
        Pre-extracted features from test_bias_node (includes ticker column)
    targets_df : pd.DataFrame
        Pre-extracted targets from test_bias_node
    """
    from ensemble.portfolio_tester import calculate_strategy_returns_from_positions, calculate_log_returns_from_candles

    print(f"\n  Generating QuantStats tearsheet for {node_name}...")

    try:
        if features_df is None or targets_df is None:
            print(f"  [FAIL] No pre-extracted features/targets provided")
            return None

        # Get best feature from permutation results
        best_feature = _get_best_feature_from_permutation(perm_df)
        if best_feature is None:
            # Fallback to first feature column
            feature_cols = [c for c in features_df.columns if c != 'ticker']
            best_feature = feature_cols[0] if feature_cols else None
        
        if best_feature is None or best_feature not in features_df.columns:
            print(f"  [FAIL] Could not find feature column")
            return None
        
        print(f"    Using feature: {best_feature}")

        # Create and fit binning model on FULL ensemble data
        if is_continuous:
            binning_model = QuantileBinningModel(n_bins=10, selection_metric='sortino', strategy='long')
        else:
            binning_model = TwoBinBinningModel(selection_metric='sortino', strategy='long')

        # Get feature and target data (all tickers)
        feature_series = features_df[best_feature].copy()
        feature_series.name = best_feature
        target_series = targets_df['log_return'].copy()
        
        # Fit binning model on full ensemble
        binning_model.fit(feature_series, target_series)
        print(f"    Model fitted on ensemble - best_long_bin: {binning_model.best_long_bin_}")

        # Filter to ES-only for predictions
        # Note: ticker column may be string 'ES' or enum Ticker.ES
        es_mask = features_df['ticker'].astype(str) == 'ES'
        es_features = features_df[es_mask][best_feature].copy()
        es_features.name = best_feature
        es_targets = targets_df[es_mask]['log_return'].copy()
        
        print(f"    ES data points: {len(es_features)}")

        # Generate predictions for ES using fitted model
        predictions = binning_model.predict(es_features, strategy='long')

        # Create positions DataFrame for ES
        # Remove timezone info from datetime to avoid merge issues
        positions_datetime = predictions.index
        if hasattr(positions_datetime, 'tz') and positions_datetime.tz is not None:
            positions_datetime = positions_datetime.tz_localize(None)
        
        positions_df = pd.DataFrame({
            'datetime': positions_datetime,
            'ticker': Ticker.ES,
            'position_fraction': predictions.values.astype(float)
        })

        # Load ES candles for return calculation (clean, without ms offset)
        es_candles_df = helpers.load_data(
            ticker=Ticker.ES,
            timeframe=TimeFrame.D,
            start=START_DATE,
            end=END_DATE
        )
        es_candles_df['ticker'] = Ticker.ES
        es_candles_df['timeframe'] = TimeFrame.D

        # Calculate strategy returns
        strategy_returns = calculate_strategy_returns_from_positions(
            positions_df, es_candles_df, strategy='long'
        )

        # Calculate baseline (buy & hold) returns
        baseline_returns = calculate_log_returns_from_candles(es_candles_df)

        # Print summary stats
        signal_freq = predictions.mean()
        print(f"    Signal frequency: {signal_freq:.1%} ({int(predictions.sum())} days active)")
        print(f"    Strategy return: {strategy_returns.sum():.2%}")
        print(f"    Baseline return: {baseline_returns.sum():.2%}")

        # Generate tearsheet
        node_dir = reports_dir / node_name.lower().replace(' ', '_').replace('%', 'pct')
        node_dir.mkdir(parents=True, exist_ok=True)
        tearsheet_path = str(node_dir / f"{node_name.lower().replace(' ', '_')}_tearsheet.html")

        generate_tearsheet(
            strategy_returns=strategy_returns,
            baseline_returns=baseline_returns,
            feature_name=f"{node_name} vs Buy & Hold (ES)",
            output_file=tearsheet_path,
            mode='html'
        )

        print(f"  [OK] Tearsheet saved to: {tearsheet_path}")
        return tearsheet_path

    except Exception as e:
        print(f"  [FAIL] Tearsheet generation failed: {e}")
        import traceback
        traceback.print_exc()
        return None

---
## Run All Tests

This cell iterates through all bias nodes in `BIAS_NODES_TO_TEST` and generates reports.

In [6]:
# Run tests for all bias nodes
all_results = {}
tearsheet_paths = {}
total_extract_time = 0

print("="*70)
print(f"STARTING BIAS NODE RESEARCH - {len(BIAS_NODES_TO_TEST)} nodes to test")
print(f"Mode: {'CACHED' if USE_CACHE else 'STREAMING'}")
print("="*70)

t_start = time.time()

for i, node_config in enumerate(BIAS_NODES_TO_TEST, 1):
    node_name = node_config['name']
    bias_spec = node_config['spec']
    is_continuous = node_config['is_continuous']
    
    print(f"\n[{i}/{len(BIAS_NODES_TO_TEST)}] Processing {node_name}...")
    
    # Generate feature analysis report
    results = test_bias_node(
        bias_spec=bias_spec,
        node_name=node_name,
        is_continuous=is_continuous,
        show_plots=False,
        export=True,
        use_cache=USE_CACHE
    )
    all_results[node_name] = results
    
    if results is not None and 'extract_time' in results:
        total_extract_time += results['extract_time']
    
    # Generate QuantStats tearsheet if enabled
    # Pass pre-extracted features/targets to ensure consistency with feature analysis
    if GENERATE_TEARSHEETS and results is not None:
        perm_df = results.get('permutation_test', None)
        features_df = results.get('features_df', None)
        targets_df = results.get('targets_df', None)
        
        tearsheet_path = generate_node_tearsheet(
            bias_spec=bias_spec,
            node_name=node_name,
            is_continuous=is_continuous,
            perm_df=perm_df,
            features_df=features_df,
            targets_df=targets_df
        )
        tearsheet_paths[node_name] = tearsheet_path

t_total = time.time() - t_start

print("\n" + "="*70)
print("ALL TESTS COMPLETE")
print(f"Total time: {t_total:.1f}s")
print(f"Feature extraction time: {total_extract_time:.1f}s")
print(f"Mode: {'CACHED' if USE_CACHE else 'STREAMING'}")
print("="*70)

STARTING BIAS NODE RESEARCH - 13 nodes to test
Mode: CACHED

[1/13] Processing Adaptive_RSI...

Testing: Adaptive_RSI [CACHED]

[1/3] Extracting features...
  [OK] Features shape: (20294, 9)
  [OK] Targets shape: (20294, 5)
  [OK] Feature columns: 8
  [OK] Extraction time: 1.66s

[2/3] Building metadata...
  [OK] FeatureExplorer created with 4 features, 20294 samples

[3/3] Generating summary report...
[DEBUG] analyze_parameter: 'adaptiversi_signal_D_maxPeriod_10_minPeriod_2_volatilityPeriod_20' samples=20294 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'adaptiversi_signal_D_maxPeriod_10_minPeriod_3_volatilityPeriod_20' samples=20294 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'adaptiversi_signal_D_maxPeriod_14_minPeriod_2_volatilityPeriod_20' samples=20294 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'adaptiversi_signal_D_maxPeriod_14_minPeriod_3_volatilityPeriod_20' samples=20294 (feature NaNs=0, target NaNs=0)
Building parameter grid for mo

---
## Summary of Results

In [7]:
# Compile summary of all results
print("\n" + "="*70)
print("SUMMARY: Bias Node Research Results")
print("="*70)

summary_data = []
for name, result in all_results.items():
    # Check for permutation_test (correct key name)
    if result is not None and 'permutation_test' in result and result['permutation_test'] is not None:
        perm_df = result['permutation_test']
        if len(perm_df) > 0:
            # Get best feature by p-value (key might be 'pval' or 'p_value')
            pval_col = 'pval' if 'pval' in perm_df.columns else 'p_value'
            best_row = perm_df.loc[perm_df[pval_col].idxmin()]
            best_pval = best_row[pval_col]
            summary_data.append({
                'Node': name,
                'Best Feature': best_row['feature'][:40] + '...' if len(best_row['feature']) > 40 else best_row['feature'],
                'P-Value': best_pval,
                'Significant': 'YES' if best_pval <= 0.1 else 'NO',
                'Status': 'OK'
            })
        else:
            summary_data.append({'Node': name, 'Status': 'No permutation results'})
    else:
        summary_data.append({'Node': name, 'Status': 'FAILED' if result is None else 'No results'})

summary_df = pd.DataFrame(summary_data)
print("\n")
print(summary_df.to_string(index=False))

# Count significant features
if 'Significant' in summary_df.columns:
    n_significant = (summary_df['Significant'] == 'YES').sum()
    n_total = len(summary_df[summary_df['Status'] == 'OK'])
    print(f"\n{n_significant}/{n_total} nodes have statistically significant features (p <= 0.1)")

print(f"\nAll reports saved to: {reports_dir}")


SUMMARY: Bias Node Research Results


            Node                                Best Feature  P-Value Significant Status
    Adaptive_RSI adaptiversi_signal_D_maxPeriod_10_minPer...     0.01         YES     OK
         Casey_C caseyc_signal_D_changePeriod_10_rankingP...     0.02         YES     OK
    Cyclical_RSI cyclicalrsi_signal_D_longPeriod_20_rsiPe...     0.20          NO     OK
   Detrended_RSI detrendedrsi_signal_D_detrendPeriod_50_r...     0.04         YES     OK
      DeMark_REI                       rei_signal_D_period_8     0.02         YES     OK
  RSI_Percentile rsipercentile_signal_D_percentilePeriod_...     0.04         YES     OK
  Stochastic_RSI stochrsi_signal_D_rsiPeriod_14_smoothK_3...     0.05         YES     OK
             TSI tsi_signal_D_longPeriod_25_shortPeriod_1...     0.23          NO     OK
      ZScore_RSI zscorersi_signal_D_rsiPeriod_14_zscorePe...     0.03         YES     OK
       Percent_B percentb_signal_D_lowerThreshold_0.0_per...     1.00  

In [8]:
# List all generated report files
import os

print("\nGenerated Report Files:")
print("="*70)

for root, dirs, files in os.walk(reports_dir):
    level = root.replace(str(reports_dir), '').count(os.sep)
    indent = ' ' * 2 * level
    folder_name = os.path.basename(root)
    if folder_name:
        print(f"{indent}[DIR] {folder_name}/")
    for file in files:
        print(f"{indent}  - {file}")


Generated Report Files:
[DIR] bias_nodes/
  [DIR] adaptive_rsi/
    - adaptive_rsi_tearsheet.html
    - all_deciles_combined.png
    - all_distributions_combined.png
    - all_signal_cumsum_combined.png
    - all_timeseries_combined.png
    - decile_bin_data.txt
    - feature_correlations.png
    - feature_data.csv
    - permutation_test_results.csv
    - summary_report.txt
    [DIR] parameter_2d_surface/
      - adaptiversi_maxPeriod_minPeriod.html
    [DIR] parameter_sensitivity/
      - adaptiversi_maxPeriod.html
  [DIR] casey_c/
    - all_deciles_combined.png
    - all_distributions_combined.png
    - all_signal_cumsum_combined.png
    - all_timeseries_combined.png
    - casey_c_tearsheet.html
    - decile_bin_data.txt
    - feature_correlations.png
    - feature_data.csv
    - permutation_test_results.csv
    - summary_report.txt
    [DIR] parameter_2d_surface/
      - caseyc_changePeriod_rankingPeriod.html
    [DIR] parameter_sensitivity/
      - caseyc_changePeriod.html
  [DIR]

---
## QuantStats Tearsheet Links

Open these HTML files in your browser to view detailed performance metrics.

In [9]:
if GENERATE_TEARSHEETS and tearsheet_paths:
    print("\nQuantStats Tearsheets Generated:")
    print("="*70)
    for node_name, path in tearsheet_paths.items():
        if path:
            print(f"  {node_name}: {path}")
else:
    print("No tearsheets generated (GENERATE_TEARSHEETS=False or all failed)")


QuantStats Tearsheets Generated:
  Adaptive_RSI: c:\Users\adabla\Trading-Algo\research\reports\bias_nodes\adaptive_rsi\adaptive_rsi_tearsheet.html
  Casey_C: c:\Users\adabla\Trading-Algo\research\reports\bias_nodes\casey_c\casey_c_tearsheet.html
  Cyclical_RSI: c:\Users\adabla\Trading-Algo\research\reports\bias_nodes\cyclical_rsi\cyclical_rsi_tearsheet.html
  Detrended_RSI: c:\Users\adabla\Trading-Algo\research\reports\bias_nodes\detrended_rsi\detrended_rsi_tearsheet.html
  DeMark_REI: c:\Users\adabla\Trading-Algo\research\reports\bias_nodes\demark_rei\demark_rei_tearsheet.html
  RSI_Percentile: c:\Users\adabla\Trading-Algo\research\reports\bias_nodes\rsi_percentile\rsi_percentile_tearsheet.html
  Stochastic_RSI: c:\Users\adabla\Trading-Algo\research\reports\bias_nodes\stochastic_rsi\stochastic_rsi_tearsheet.html
  TSI: c:\Users\adabla\Trading-Algo\research\reports\bias_nodes\tsi\tsi_tearsheet.html
  ZScore_RSI: c:\Users\adabla\Trading-Algo\research\reports\bias_nodes\zscore_rsi\zscor